# 6. German Language NLP & Regional AI Considerations

This notebook focuses on German NLP challenges and localized AI engineering:
1. **German Text Processing & Tokenization Efficiency** (Umlauts `ä, ö, ü, ß`, BPE token compression)
2. **Compound Word Splitting (Komposita Splitting)**
3. **Formal vs Informal Register Conversion (`Sie` vs `du`)**
4. **German LLM Evaluation Benchmarks & DACH Region Context**

In [ ]:
import re
import json
from typing import List, Dict, Tuple, Any

print("=== German Language AI Setup Complete ===")

## Step 1: German Tokenizer Efficiency & Character Normalization

German features long compound words and extended character sets (`ä, ö, ü, ß`).
Standard subword tokenizers (like Byte-Pair Encoding trained primarily on English) often fragment German text into excessively small subword tokens, leading to higher token consumption and latency.

In [ ]:
class GermanTokenEfficiencyAnalyzer:
    @staticmethod
    def calculate_token_stats(english_text: str, german_text: str) -> Dict[str, Any]:
        # Character count comparison
        char_ratio = len(german_text) / len(english_text) if english_text else 1.0
        
        # Word count comparison
        en_words = len(english_text.split())
        de_words = len(german_text.split())
        
        # Simulated subword BPE token count (German subword splitting overhead factor ~1.35x)
        simulated_en_bpe_tokens = int(en_words * 1.2)
        simulated_de_bpe_tokens = int(de_words * 1.6) # German higher subword fragmentation
        
        return {
            "en_words": en_words,
            "de_words": de_words,
            "en_bpe_est": simulated_en_bpe_tokens,
            "de_bpe_est": simulated_de_bpe_tokens,
            "token_overhead_ratio": round(simulated_de_bpe_tokens / simulated_en_bpe_tokens, 2)
        }

en_sample = "Retrieval-augmented generation increases model accuracy for enterprise search applications."
de_sample = "Suchergebnisgestützte Generierung erhöht die Modellgenauigkeit für Unternehmenssuchanwendungen."

stats = GermanTokenEfficiencyAnalyzer.calculate_token_stats(en_sample, de_sample)
print("Language Token Efficiency Analysis:")
print(f" English Words: {stats['en_words']} | Est. Subword Tokens: {stats['en_bpe_est']}")
print(f" German Words:  {stats['de_words']} | Est. Subword Tokens: {stats['de_bpe_est']}")
print(f" -> German Subword Token Overhead Ratio: {stats['token_overhead_ratio']}x")

## Step 2: German Compound Word Splitting (Komposita Splitting)

German frequently forms new nouns by concatenating existing words (e.g., *Donaudampfschifffahrt*).
Splitting compound words improves search recall in vector and keyword databases.

In [ ]:
class GermanCompoundSplitter:
    def __init__(self):
        # Known root dictionary for mock splitting
        self.roots = [
            "donau", "dampf", "schiff", "fahrt", "gesellschaft", "kapitän",
            "daten", "schutz", "grund", "verordnung", "anwendung", "entwickler",
            "künstlich", "intelligenz", "sprach", "modell"
        ]

    def split(self, compound_word: str) -> List[str]:
        word_lower = compound_word.lower()
        parts = []
        current = ""
        for char in word_lower:
            current += char
            if current in self.roots:
                parts.append(current)
                current = ""
        if current:
            parts.append(current)
        return parts if len(parts) > 1 else [compound_word]

splitter = GermanCompoundSplitter()
compounds = [
    "Donaudampfschifffahrtsgesellschaftskapitän",
    "Datenschutzgrundverordnung",
    "Sprachmodell"
]

print("German Compound Word Splitting (Komposita):")
for comp in compounds:
    tokens = splitter.split(comp)
    print(f" '{comp}' -> {tokens}")

## Step 3: Register Transformation (`Sie` vs `du`)

German has two distinct forms of address:
- **Informal (`du` / `dein`)**: Used for close contacts, peer tech communication, consumer applications.
- **Formal (`Sie` / `Ihr`)**: Used for business, legal, banking, and professional DACH communications.

In [ ]:
class GermanRegisterTransformer:
    def __init__(self):
        self.du_to_sie_map = {
            "du": "Sie",
            "dich": "Sie",
            "dir": "Ihnen",
            "dein": "Ihr",
            "deine": "Ihre",
            "deinen": "Ihren",
            "kannst": "können",
            "hast": "haben"
        }

    def transform_to_formal(self, text: str) -> str:
        words = text.split()
        res = []
        for w in words:
            clean_w = re.sub(r'[^\w]', '', w.lower())
            if clean_w in self.du_to_sie_map:
                replacement = self.du_to_sie_map[clean_w]
                # Match capitalisation
                if w[0].isupper():
                    replacement = replacement.capitalize()
                res.append(w.replace(clean_w, replacement))
            else:
                res.append(w)
        return " ".join(res)

informal_text = "Kannst du mir bitte deine Unterlagen schicken, damit dir geholfen werden kann?"
transformer = GermanRegisterTransformer()
formal_text = transformer.transform_to_formal(informal_text)

print(f"Informal (du): '{informal_text}'")
print(f"Formal   (Sie): '{formal_text}'")

## Step 4: German LLM Benchmarking & Evaluation

Evaluating German models requires DACH-specific test suites measuring translation fidelity, administrative vocabulary, and cultural context.

In [ ]:
class GermanLLMBenchmark:
    @staticmethod
    def evaluate_response(prompt: str, response: str, expected_keywords: List[str]) -> Dict[str, Any]:
        resp_lower = response.lower()
        matched = [kw for kw in expected_keywords if kw.lower() in resp_lower]
        score = len(matched) / len(expected_keywords) if expected_keywords else 0.0
        return {
            "prompt": prompt,
            "match_score": round(score, 2),
            "matched_keywords": matched,
            "passed": score >= 0.7
        }

test_prompt = "Erkläre die Grundsätze der DSGVO im Kontext von KI-Systemen."
sample_model_resp = "Die DSGVO verlangt Datensparsamkeit, Zweckbindung und Transparenz bei der Verarbeitung personenbezogener Daten in KI-Modellen."
expected_terms = ["DSGVO", "Datensparsamkeit", "Zweckbindung", "Transparenz", "personenbezogener"]

eval_res = GermanLLMBenchmark.evaluate_response(test_prompt, sample_model_resp, expected_terms)
print(f"German Benchmark Result: Passed={eval_res['passed']} (Score: {eval_res['match_score']*100:.0f}%)")
print(f"Matched Regulatory Terms: {eval_res['matched_keywords']}")